# Zonal statistics — Landsat bands → LSOA means (full resolution)

Computes per-LSOA mean surface reflectance for each Landsat 8 band at full ~30 m resolution using `exactextract`, which weights each pixel by the fraction of its area covered by the polygon (more accurate than point-in-polygon for small areas like LSOAs).

Output is written to `jupyterlite/content/landsat_features.csv` for bundling with the workshop site.

In [1]:
from pathlib import Path
import pandas
import geopandas
import xarray as xr
from exactextract import exact_extract

repo = Path('../../../../')
content = repo / 'jupyterlite/content'

In [2]:
sr = xr.open_dataset(content / 'london_landsat.nc', engine='scipy')['SR']
lsoa = geopandas.read_file(content / 'uk_lsoa_london_embeds_2020.geojson')[['LSOA21CD', 'geometry']]
print(sr)
print(f'{len(lsoa)} LSOAs')

<xarray.DataArray 'SR' (band: 7, y: 1141, x: 2314)> Size: 148MB
[18481918 values with dtype=float64]
Coordinates:
  * band     (band) object 56B 'B1' 'B2' 'B3' 'B4' 'B5' 'B6' 'B7'
  * y        (y) float64 9kB 51.7 51.7 51.7 51.7 ... 51.28 51.28 51.28 51.27
  * x        (x) float64 19kB -0.5196 -0.5192 -0.5189 ... 0.3434 0.3438 0.3441
Attributes:
    AREA_OR_POINT:  Area
4951 LSOAs


In [6]:
features = exact_extract(
    sr,
    lsoa,
    ["mean", "stdev"],
    include_cols=["LSOA21CD"],
    output="pandas",
).set_index("LSOA21CD")
features

,band_1_mean,band_2_mean,band_3_mean,band_4_mean,band_5_mean,band_6_mean,band_7_mean,band_1_stdev,band_2_stdev,band_3_stdev,band_4_stdev,band_5_stdev,band_6_stdev,band_7_stdev
LSOA21CD,,,,,,,,,,,,,,
E01000001,9045.957598,9337.677284,9932.626638,10032.516547,11766.852845,11331.602611,10413.945508,719.210355,726.906616,916.237082,995.732633,1968.040788,1319.943135,1035.664347
E01000002,9089.642507,9434.199915,10026.246851,10154.483337,11394.728361,11309.325513,10423.907225,867.869501,985.225290,1136.623901,1319.459233,1694.360574,1732.565521,1463.281783
E01000003,9325.231946,9597.432820,10173.861357,10297.926992,12314.237093,11807.224375,10722.776000,656.064862,695.961177,795.186641,924.491028,1684.652027,1200.000284,927.227854
E01000005,9334.111201,9595.257820,10106.214794,10280.362138,11393.827715,11487.028916,10794.883461,566.562775,621.121642,729.772513,811.499388,1254.313406,1117.264708,906.458602
E01000006,9043.150208,9262.264618,9857.878024,10060.270729,13122.604067,12394.307425,11180.170006,246.157792,252.614834,298.299233,387.706154,789.145104,514.343472,451.983295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
E01035718,8834.106033,8899.731060,9579.749231,9400.103372,15879.255450,13310.011736,10763.262566,463.973230,547.981089,715.509471,891.275498,2918.394323,2413.690638,1581.393058
E01035719,9094.324665,9324.799868,9860.220551,9920.148114,12020.153311,11336.205719,10424.654278,599.694505,705.368757,879.754751,984.647272,1342.389783,1072.400448,911.299689
E01035720,9269.982468,9558.194222,10211.073999,10346.162639,12785.272022,11929.465696,10921.531893,938.556633,1110.226993,1359.483733,1553.043844,1730.836502,1427.408058,1237.073898


In [7]:
out_path = content / 'landsat_features.csv'
features.to_csv(out_path)
print(f'Written {len(features)} rows to {out_path}')

Written 4951 rows to ../../../../jupyterlite/content/landsat_features.csv
